In [1]:
!pip install open_clip_torch --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 31.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00


In [2]:
!nvidia-smi

Sun Mar 22 13:40:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torchinfo import summary
import torch.nn as nn
from torchvision.datasets import ImageFolder
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import _LRScheduler, ReduceLROnPlateau
import torch.utils.data as data
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import random
import timm
import cv2

In [4]:
SEED = 1234
VISUALIZE = False
DEBUG = False

patch_size = 112
stride = 56
device="cuda"

alpha_feat = torch.nn.Parameter(torch.tensor(0.5, device=device))
alpha_proto = torch.nn.Parameter(torch.tensor(0.5, device=device))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [5]:
import os
import torch
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import numpy as np

batch_size = 16

# -----------------------
# 1. TRANSFORMS
# -----------------------
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=25, translate=(0.25, 0.25), scale=(0.75, 1.25)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# -----------------------
# 2. LOAD DATASET (NO TRANSFORM HERE)
# -----------------------
root_dir = "/kaggle/input/datasets/wenewone/cub2002011/CUB_200_2011/images"

full_dataset = datasets.ImageFolder(root_dir, transform=None)

class_names = full_dataset.classes
print("Classes:", class_names)

N = len(full_dataset)
print("Total images =", N)

# -----------------------
# 3. CREATE SPLITS (INDICES)
# -----------------------
generator = torch.Generator().manual_seed(123)
indices = torch.randperm(N, generator=generator)

train_size = int(0.6 * N)
val_size   = int(0.2 * N)

train_idx = indices[:train_size]
val_idx   = indices[train_size:train_size + val_size]
test_idx  = indices[train_size + val_size:]

# -----------------------
# 4. CREATE SEPARATE DATASETS
# -----------------------
train_dataset = Subset(
    datasets.ImageFolder(root_dir, transform=train_transform),
    train_idx
)

val_dataset = Subset(
    datasets.ImageFolder(root_dir, transform=val_test_transform),
    val_idx
)

test_dataset = Subset(
    datasets.ImageFolder(root_dir, transform=val_test_transform),
    test_idx
)

# -----------------------
# 5. DATALOADERS
# -----------------------
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# -----------------------
# 6. PRINT STATS
# -----------------------
print(f"Train:      {len(train_dataset)} images")
print(f"Validation: {len(val_dataset)} images")
print(f"Test:       {len(test_dataset)} images")
print(f"Total check: {len(train_dataset)+len(val_dataset)+len(test_dataset)} images")

Classes: ['001.Black_footed_Albatross', '002.Laysan_Albatross', '003.Sooty_Albatross', '004.Groove_billed_Ani', '005.Crested_Auklet', '006.Least_Auklet', '007.Parakeet_Auklet', '008.Rhinoceros_Auklet', '009.Brewer_Blackbird', '010.Red_winged_Blackbird', '011.Rusty_Blackbird', '012.Yellow_headed_Blackbird', '013.Bobolink', '014.Indigo_Bunting', '015.Lazuli_Bunting', '016.Painted_Bunting', '017.Cardinal', '018.Spotted_Catbird', '019.Gray_Catbird', '020.Yellow_breasted_Chat', '021.Eastern_Towhee', '022.Chuck_will_Widow', '023.Brandt_Cormorant', '024.Red_faced_Cormorant', '025.Pelagic_Cormorant', '026.Bronzed_Cowbird', '027.Shiny_Cowbird', '028.Brown_Creeper', '029.American_Crow', '030.Fish_Crow', '031.Black_billed_Cuckoo', '032.Mangrove_Cuckoo', '033.Yellow_billed_Cuckoo', '034.Gray_crowned_Rosy_Finch', '035.Purple_Finch', '036.Northern_Flicker', '037.Acadian_Flycatcher', '038.Great_Crested_Flycatcher', '039.Least_Flycatcher', '040.Olive_sided_Flycatcher', '041.Scissor_tailed_Flycatcher',

In [6]:
import torch

torch.backends.cudnn.enabled = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

In [7]:
import torch
import torch.nn.functional as F
import open_clip
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# 1. LOAD OPENAI CLIP
# -------------------------
model, preprocess_train, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-16",
    pretrained="openai"
)

model = model.to(device)
model.eval()

tokenizer = open_clip.get_tokenizer("ViT-B-16")

cnn = models.efficientnet_b0(pretrained=True)

# remove classification layer
cnn = nn.Sequential(*list(cnn.children())[:-1])

cnn = cnn.to(device)
cnn.eval()

cnn_proj_layer = nn.Linear(1280, 512).to(device)

# Freeze CLIP 
for p in model.parameters():
    p.requires_grad = False

# Train CNN 
for p in cnn.parameters():
    p.requires_grad = True

print("Model device:", next(model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 142MB/s]


Model device: cuda:0
CUDA available: True
GPU name: Tesla T4


In [8]:
#--------------------------#
#-----Helper Functions-----#
#--------------------------#

import matplotlib.pyplot as plt

def show_tensor(img, title="image"):
    img = img.squeeze().permute(1,2,0).cpu().numpy()
    img = (img - img.min())/(img.max()-img.min()+1e-8)

    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

def visualize_patches(image, patch_size=32):

    img = image.squeeze().permute(1,2,0).cpu().numpy()
    h, w, _ = img.shape

    fig, ax = plt.subplots(1, figsize=(5,5))
    ax.imshow(img)

    for i in range(0,h,patch_size):
        ax.axhline(i,color='red',linewidth=0.5)

    for j in range(0,w,patch_size):
        ax.axvline(j,color='red',linewidth=0.5)

    ax.set_title("Patch Grid")
    ax.axis("off")
    plt.show()

def overlay_attention(image, attn_map):

    img = image.squeeze().permute(1,2,0).cpu().numpy()

    heatmap = cv2.resize(attn_map,(224,224))
    heatmap = cv2.applyColorMap((heatmap*255).astype(np.uint8), cv2.COLORMAP_JET)

    overlay = heatmap*0.5 + img*255*0.5

    plt.figure(figsize=(4,4))
    plt.imshow(overlay.astype(np.uint8))
    plt.title("Attention Overlay")
    plt.axis("off")
    plt.show()

In [9]:
def encode_text(classnames):

    templates = [
        "a photo of a {}",
        "a photo of the {}",
        "a close-up photo of a {}",
        "a photo of a small bird, a {}",
        "a bird species called {}"
    ]

    all_class_embeddings = []

    for c in classnames:

        c = c.replace("_", " ")

        prompts = [t.format(c) for t in templates]

        tokens = tokenizer(prompts).to(device)

        with torch.no_grad():
            text_features = model.encode_text(tokens)

        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # average template embeddings
        text_features = text_features.mean(dim=0)

        all_class_embeddings.append(text_features)

    text_features = torch.stack(all_class_embeddings)

    return text_features

In [10]:
def encode_image_fused(image):

    clip_feat = model.encode_image(image)
    clip_feat = F.normalize(clip_feat, dim=-1)

    cnn_feat = cnn(image).view(image.size(0), -1)
    cnn_feat = F.normalize(cnn_feat, dim=-1)

    cnn_proj = cnn_proj_layer(cnn_feat)   # 2048 → 512
    cnn_proj = F.normalize(cnn_proj, dim=-1)

    alpha = torch.nn.Parameter(torch.tensor(0.5)).to(device)
    fused = clip_feat + alpha_feat * cnn_proj

    fused = F.normalize(fused, dim=-1)

    return fused

In [11]:
def pass1_clip(image, text_features):

    # fused embedding
    image_features = encode_image_fused(image)

    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    # similarity with text embeddings
    logits = image_features @ text_features.T

    probs = logits.softmax(dim=-1)

    return logits, probs, image_features

In [12]:
def get_topk(logits, k=5):
    values, indices = torch.topk(logits, k)
    return indices.squeeze().tolist()

In [13]:
def get_patch_embeddings(image):

    img = image.squeeze().permute(1,2,0).cpu().numpy()
    h, w, c = img.shape

    patch_size = 112
    stride = 56

    patches = []

    for i in range(0, h - patch_size + 1, stride):
        for j in range(0, w - patch_size + 1, stride):

            patch = img[i:i+patch_size, j:j+patch_size]

            patch = (patch * 255).astype(np.uint8)
            patch = Image.fromarray(patch)

            patch = preprocess(patch).unsqueeze(0).to(device)

            with torch.no_grad():
                feat = encode_image_fused(patch)

            feat = feat / feat.norm(dim=-1, keepdim=True)

            patches.append(feat)

    patches = torch.cat(patches, dim=0)

    return patches

In [14]:
def compute_attention_map(patch_feats, text_feat, class_name=None):

    patch_feats = F.normalize(patch_feats, dim=-1)
    text_feat = F.normalize(text_feat, dim=-1)

    attn = patch_feats @ text_feat.T
    attn = attn.squeeze().cpu().numpy()

    grid_h = (224 - patch_size) // stride + 1
    grid_w = (224 - patch_size) // stride + 1
    
    attn = attn.reshape(grid_h, grid_w)

    attn = (attn - attn.min())/(attn.max()-attn.min()+1e-8)

    if VISUALIZE:
        # ---- visualization ----
        plt.figure(figsize=(8,3))
    
        # patch attention map
        plt.subplot(1,2,1)
        plt.imshow(attn, cmap="jet")
        plt.title(f"Patch Attention\n{class_name}")
        plt.colorbar()
        plt.axis("off")

    # resized attention (image scale)
    attn_resized = cv2.resize(attn, (224,224))

    if VISUALIZE:
        plt.subplot(1,2,2)
        plt.imshow(attn_resized, cmap="jet")
        plt.title("Attention (Resized)")
        plt.colorbar()
        plt.axis("off")
    
        plt.show()

    return attn

In [15]:
def crop_attention_region(image, attn_map, threshold=0.6):

    attn_map = cv2.resize(attn_map, (224,224))

    mask = attn_map > threshold

    coords = np.column_stack(np.where(mask))

    if len(coords) == 0:
        return image

    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0)

    img_np = image.squeeze().permute(1,2,0).cpu().numpy()

    crop = img_np[y0:y1, x0:x1]

    crop = cv2.resize(crop, (224,224))

    crop = torch.from_numpy(crop).permute(2,0,1).unsqueeze(0).float()
    crop = crop.to(device)

    return crop

In [16]:
def pass2_crop(crop, text_feat):

    crop = crop.to(device)

    with torch.no_grad():
        crop_feat = encode_image_fused(crop)

    crop_feat = crop_feat / crop_feat.norm(dim=-1, keepdim=True)

    score = crop_feat @ text_feat.T

    return score.item()

In [17]:
def fuse_scores(global_score, refined_score, alpha=0.7):
    return alpha * global_score + (1-alpha) * refined_score

In [18]:
def soft_mask_image(image, attn_map, min_keep=0.1):
    """
    Apply soft attention masking instead of cropping.
    
    image: tensor [1,3,224,224]
    attn_map: [grid,grid] attention
    """

    # resize attention to image size
    attn_resized = cv2.resize(attn_map, (224,224))

    # keep some background so context is not lost
    attn_resized = np.clip(attn_resized, min_keep, 1.0)

    # convert to tensor
    mask = torch.from_numpy(attn_resized).float().to(image.device)

    mask = mask.unsqueeze(0).unsqueeze(0)  # [1,1,224,224]

    # broadcast across RGB channels
    masked_image = image * mask

    return masked_image

In [19]:
def prototype_logits(image_features, prototypes):

    prototypes = F.normalize(prototypes, dim=-1)
    image_features = F.normalize(image_features, dim=-1)

    logits = image_features @ prototypes.T

    return logits

In [20]:
def class_conditional_rerank(
    image,
    text_features,
    class_names,
    prototypes,
    K=5   # top-K from ALL classes
):

    # ---- CLIP pass (ALL classes) ----
    logits, probs, image_features = pass1_clip(
        image,
        text_features
    )

    proto_logits = prototype_logits(image_features, prototypes)
    
    logits = logits + alpha_proto * proto_logits

    # ---- get top-K GLOBAL classes ----
    topk = torch.topk(logits, K, dim=1).indices[0]

    patch_feats = get_patch_embeddings(image)

    final_scores = {}

    for k in topk:

        class_idx = k.item()

        text_feat = text_features[class_idx].unsqueeze(0)

        # ---- attention ----
        attn = compute_attention_map(patch_feats, text_feat)

        masked = soft_mask_image(image, attn)

        refined = pass2_crop(masked, text_feat)

        global_score = logits[0, class_idx].item()

        final = fuse_scores(global_score, refined)

        final_scores[class_idx] = final

    pred = max(final_scores, key=final_scores.get)

    return pred, final_scores

In [21]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#Metric Computation 
def compute_metrics(y_true, y_pred, average='macro'):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)
    return acc, prec, rec, f1

In [22]:
import random
from collections import defaultdict

# -------------------------
# K-SHOT DATASET
# -------------------------
def create_k_shot_dataset(dataset, K=5):
    class_indices = defaultdict(list)

    for i, (_, label) in enumerate(dataset):
        class_indices[label].append(i)

    selected_indices = []

    for cls, indices in class_indices.items():
        if len(indices) >= K:
            selected = random.sample(indices, K)
        else:
            selected = random.choices(indices, k=K)

        selected_indices.extend(selected)

    return Subset(dataset, selected_indices)


# -------------------------
# GLOBAL PROTOTYPES
# -------------------------
def build_global_prototypes(model, dataset, num_classes):    
    
    model.eval()  

    print("Prototype function called")

    class_feats = [[] for _ in range(num_classes)]

    with torch.no_grad():
        for img, label in dataset:
            img = img.unsqueeze(0).to(device)

            feat = encode_image_fused(img)

            class_feats[label].append(feat.squeeze())

    prototypes = []

    for c in range(num_classes):
        if len(class_feats[c]) == 0:
            proto = torch.zeros(512, device=device)
        else:
            feats = torch.stack(class_feats[c])
            proto = feats.mean(dim=0)
    
        proto = F.normalize(proto, dim=-1)
        prototypes.append(proto)

    prototypes = torch.stack(prototypes)
    prototypes = F.normalize(prototypes, dim=-1)

    return prototypes.to(device)


# -------------------------
# TRAIN (FIXED)
# -------------------------
def train_global(model, train_loader, text_features, optimizer, prototypes):

    model.train()
    cnn.train()
    cnn_proj_layer.train()
    
    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        feats = encode_image_fused(images)

        clip_logits = (feats @ text_features.T) * 100

        # ---- correct usage ----
        if prototypes is not None:
            proto_logits = prototype_logits(feats, prototypes)
            logits = clip_logits + alpha_proto * proto_logits
        else:
            logits = clip_logits

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


# -------------------------
# EVALUATE (UNCHANGED)
# -------------------------

def evaluate(
    model,
    loader,
    text_features,
    prototypes,
    class_names,
    K=5
):

    model.eval()
    cnn.eval()
    cnn_proj_layer.eval()

    all_preds = []
    all_labels = []

    print("Evaluate Called")

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            for i in range(images.size(0)):

                image = images[i].unsqueeze(0)
                label = labels[i].item()

                pred, _ = class_conditional_rerank(
                    image,
                    text_features,
                    class_names,
                    prototypes,
                    K=K
                )

                all_preds.append(pred)
                all_labels.append(label)

                if len(all_preds) % 50 == 0:
                    acc_so_far = accuracy_score(all_labels, all_preds)
                    print(f"Processed {len(all_preds)} samples | Acc so far: {acc_so_far:.4f}")

    # ---- compute metrics ----
    acc, prec, rec, f1 = compute_metrics(all_labels, all_preds)

    print("\nFinal Metrics:")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")

    return acc, prec, rec, f1


# -------------------------
# MAIN
# -------------------------
K = 10

train_dataset_k = create_k_shot_dataset(train_dataset, K)
train_loader_k = DataLoader(train_dataset_k, batch_size=16, shuffle=True)

num_classes = len(class_names)

optimizer = optim.Adam(
    list(cnn.parameters()) +
    list(cnn_proj_layer.parameters()) +
    [alpha_feat, alpha_proto],
    lr=1e-3
)

# ---- TEXT FEATURES ----
text_features = encode_text(class_names)
text_features = F.normalize(text_features, dim=-1)
text_features = text_features.to(device)

prototypes = None

for epoch in range(20):

    # build prototypes AFTER some learning
    if epoch == 5:
        print("Building prototypes...")
        prototypes = build_global_prototypes(
            model,
            train_dataset_k,
            num_classes
        )

    loss = train_global(
        model,
        train_loader_k,
        text_features,
        optimizer,
        prototypes
    )

    print(f"Epoch {epoch}: Loss={loss:.4f}")

# ---- PROTOTYPES ----
prototypes = build_global_prototypes(model, train_dataset_k, num_classes)

# ---- EVALUATION ----
test_acc = evaluate(model, test_loader, text_features, prototypes, class_names)

Epoch 0: Loss=2.3874
Epoch 1: Loss=1.9290
Epoch 2: Loss=1.7186
Epoch 3: Loss=1.5176
Epoch 4: Loss=1.3654
Building prototypes...
Prototype function called
Epoch 5: Loss=1.2130
Epoch 6: Loss=1.0527
Epoch 7: Loss=0.8822
Epoch 8: Loss=0.8039
Epoch 9: Loss=0.7015
Epoch 10: Loss=0.6608
Epoch 11: Loss=0.5441
Epoch 12: Loss=0.4747
Epoch 13: Loss=0.5078
Epoch 14: Loss=0.3987
Epoch 15: Loss=0.3554
Epoch 16: Loss=0.3775
Epoch 17: Loss=0.3137
Epoch 18: Loss=0.2536
Epoch 19: Loss=0.2843
Prototype function called
Evaluate Called
Processed 50 samples | Acc so far: 0.7800
Processed 100 samples | Acc so far: 0.7100
Processed 150 samples | Acc so far: 0.6467
Processed 200 samples | Acc so far: 0.6400
Processed 250 samples | Acc so far: 0.6240
Processed 300 samples | Acc so far: 0.5933
Processed 350 samples | Acc so far: 0.5914
Processed 400 samples | Acc so far: 0.5900
Processed 450 samples | Acc so far: 0.5911
Processed 500 samples | Acc so far: 0.5980
Processed 550 samples | Acc so far: 0.6018
Process